# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

In [2]:
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)

NameError: name 'pd' is not defined

Ваши ответы и объяснения:

1. 20, "a:10"
2. {"Алматы": 5, "Астана": 4, "Шымкент": 2, "Шымкент": 1}
3. "x": [2,3]
4. (2,2)
5. [1,2], 5

In [3]:
import pandas as pd

In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [6]:
df.head(10)

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
5,102231,2026-09-01 09:00,C00763,Шымкент,Книги,3670.0,1.0,отменён
6,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
7,101963,2026-09-01 09:00,C01730,Астана,Одежда,9920.0,1.0,оплачен
8,100172,2026-09-01 09:00,C00896,Алматы,Книги,3500.0,2.0,оплачен
9,101280,2026-09-01 09:00,C01349,Астана,Электроника,18490.0,2.0,оплачен


In [7]:
df.describe()

,order_id,amount,delivery_days
count,5463.000000,5.463000e+03,5303.000000
mean,102598.220209,2.011411e+04,2.353196
std,1524.574180,2.343251e+05,1.468226
min,100001.000000,7.200000e+02,1.000000
25%,101277.500000,5.870000e+03,1.000000
50%,102557.000000,1.032000e+04,2.000000
75%,103917.500000,1.869500e+04,3.000000
max,105283.000000,9.999990e+06,8.000000


In [8]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [9]:
df['status'].unique()

array(['оплачен', 'отменён'], dtype=object)

In [10]:
# 1
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB


In [11]:
print("Пустых строк в днях доставки: ",df['delivery_days'].isna().sum())

Пустых строк в днях доставки:  160


In [12]:
print("Уникальных городов:", df['city'].nunique())
print("Уникальных категорий:", df['category'].nunique())
print("Уникальных статусов:", df['status'].nunique())
print("Уникальных клиентов:", df['customer_id'].nunique())

Уникальных городов: 14
Уникальных категорий: 5
Уникальных статусов: 2
Уникальных клиентов: 2003


проблемы данных которые ужы видны:
1. я думаю для дальнейшего анализа данных строки category и status должны быть int, так их будет легче классифицировать
2. 'delivery_days' имеет пустые значения, их нужно или убрать или заполнить средним значением
3.


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [13]:
filtered_df = df[paid & not_test]
display(filtered_df['amount'].describe())

,amount
count,4901.000000
mean,14672.391349
std,13249.340787
min,720.000000
25%,5880.000000
50%,10300.000000
75%,18690.000000
max,139040.000000


Типичный чек это сумма от 5880 до 18690, это межквартильный размах. В отчете лучше показать медиану, потому что ср. арифм. размах слишком большой. Медиана делит данные ровно пополам что не создает иллюзию что обычные заказы очень большие.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [14]:
# 3
unpaid = df["status"] == "отменён"
SH = ["Шымкент", "шымкент", "Shymkent"]
ALM = ["Алматы"]

total_cancelled_share = unpaid.mean()

sh_df = df[df["city"].isin(SH)]
sh_cancelled_share = (sh_df["status"] == "отменён").mean()

almaty_df = df[df["city"].isin(ALM)]
almaty_cancelled_share = (almaty_df["status"] == "отменён").mean()

print(f"Доля отмен глобально: {total_cancelled_share:.2%}")
print(f"Доля отмен в Шымкенте: {sh_cancelled_share:.2%}")
print(f"Доля отмен в Алматы: {almaty_cancelled_share:.2%}")

Доля отмен глобально: 10.23%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.99%


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [15]:
# Восстанавливаем исходный датафрейм из файла, так как переменная df была перезаписана
df = pd.read_csv('orders.csv')
df['amount'] = pd.to_numeric(df['amount'].str.replace(' ', ''))

# Фильтры
sep = df["order_date"] < "2026-10"
octo = ~sep
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"

target_orders = paid & not_test
sh_mask = df['city'].isin(SH)
alm_mask = df['city'].isin(ALM)

# --- ШЫМКЕНТ ---
sh_sep_rev = df[target_orders & sh_mask & sep]['amount'].sum()
sh_oct_rev = df[target_orders & sh_mask & octo]['amount'].sum()
sh_change = (sh_oct_rev - sh_sep_rev) / sh_sep_rev

# --- АЛМАТЫ ---
alm_sep_rev = df[target_orders & alm_mask & sep]['amount'].sum()
alm_oct_rev = df[target_orders & alm_mask & octo]['amount'].sum()
alm_change = (alm_oct_rev - alm_sep_rev) / alm_sep_rev

print(f"Шымкент | Выручка в сентябре: {sh_sep_rev:,.0f} KZT, в октябре: {sh_oct_rev:,.0f} KZT. Изменение: {sh_change:.2%}")
print(f"Алматы   | Выручка в сентябре: {alm_sep_rev:,.0f} KZT, в октябре: {alm_oct_rev:,.0f} KZT. Изменение: {alm_change:.2%}")

Шымкент | Выручка в сентябре: 4,949,800 KZT, в октябре: 4,044,010 KZT. Изменение: -18.30%
Алматы   | Выручка в сентябре: 13,638,620 KZT, в октябре: 12,975,290 KZT. Изменение: -4.86%


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [16]:
# 5
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))



TypeError: Cannot perform 'rand_' with a dtyped [object] array and scalar of type [bool]

### Исправленный код для задания 5

**Ошибки в коде от AI:**
1. **Отсутствие круглых скобок** вокруг условий сравнения. Оператор `&` имеет более высокий приоритет, чем `==`, поэтому Pandas пытается вычислить `"Шымкент" & df["status"]`, что вызывает ошибку `TypeError`.
2. **Неполный учет написания города**. В данных Шымкент пишется разными способами (например, с маленькой буквы или латиницей). Фильтрация `df["city"] == "Шымкент"` тихо теряет часть отмененных заказов.

In [17]:
# Исправленный вариант с учетом скобок и всех написаний Шымкента
SH = ["Шымкент", "шымкент", "Shymkent"]
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print("Количество отмененных заказов в Шымкенте (исправлено):", len(cancel_sh))

Количество отмененных заказов в Шымкенте (исправлено): 164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6

In [19]:
# Загружаем и подготавливаем данные, если df еще не определен
import pandas as pd
try:
    df
except NameError:
    df = pd.read_csv('orders.csv')
    df['amount'] = pd.to_numeric(df['amount'].str.replace(' ', ''))

# Находим топ-клиента по количеству заказов
top_customer = df["customer_id"].value_counts().idxmax()
order_count = df["customer_id"].value_counts().max()

print(f"Самый частый клиент: {top_customer} (Количество заказов: {order_count})\n")

# Фильтруем все заказы этого клиента и выводим нужные столбцы
customer_orders = df[df["customer_id"] == top_customer]
display(customer_orders[["order_date", "city", "amount", "status"]])

# Проверяем на дубликаты по order_id
duplicates_count = customer_orders["order_id"].duplicated().sum()
print(f"Количество повторяющихся order_id у этого клиента: {duplicates_count}")

Самый частый клиент: C01818 (Количество заказов: 11)



,order_date,city,amount,status
57,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,2026-09-24 18:00,Астана,1490.0,оплачен
2617,2026-09-29 09:00,Астана,22170.0,оплачен
2776,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,2026-10-18 08:00,Астана,13600.0,оплачен
4610,2026-10-21 19:00,Алматы,21990.0,оплачен


Количество повторяющихся order_id у этого клиента: 1


**Вывод для менеджера по заданию 6:**
- Самым частым клиентом является пользователь с ID, указанным выше.
- У этого клиента присутствуют абсолютно идентичные дубликаты по `order_id` (заказы продублированы в таблице).
- **Что это значит для выручки:** Так как строки дублируются, простое суммирование столбца `amount` искусственно завышает реальную выручку. Перед подсчетом итоговых показателей необходимо удалить дубликаты записей.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8